### Objetivo
Lê os 12 CVS Mensais do volume voebem.bronze.arquivos/vra e materializa voebem.arquivos.vra

Regras da Camada Bronze:

Nada de Tipagem: tudo string como veio no arquivo
Nada de Filtro: nenhuma linha é descastada
Colunas de auditoria: de qual arquivo veio e quando foi ingerido
Idempotente: rodar duas vezes não duplica

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra" 
#test

### Leitura
4 Opções resolvem 4 Problemas do arquivo:
\
_**opção** / problema_

**sep=";"** / separador brasileiro, não vírgula
**skipRows=1** / a 1ª linha é atualizada em , não o cabeçalho. O BOM EF BB BF mora nela e some junto
**header=true** / a 2ª linha (a primeira que sobra) é o cabeçalho de verdade
**inferSchema desligado** / bronze não tipa; tudo vem como string

In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO)

)

print("colunas lidas do arquivo:")
for c in bruto.columns:
    print(f" {c !r}")


### Problema: Nome da coluna contém espaço
CVS aceita espaço, Delta não. Para corrigir e manter as características do medalhão bronze, uma correção explícita em código deve ser feita sem causar alterações em valor ou granularidade, apenas a grafia do nome.

In [0]:
RENOMEAR = {
    "ICAO Empresa Aérea": "icao_empresa",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_origem",
    "ICAO Aeródromo Destino": "icao_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa"
    }

faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada não encontrada no CSV: {faltando}"

renomeado = bruto.select(
    *[F.col(f"{origem}").cast("string").alias(novo) for origem,
      novo in RENOMEAR.items()]
)

### Auditoria
O arquivo precisa de mais duas colunas ainda não existentes:
_arquivo_origem e _ingerido_em

In [0]:
bronze = renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")
).withColumn(
    "_ingerido_em", F.current_timestamp()
    )

### Escrita Idempotente
Fullrefres determinístico sobre o conjunto inteiro de arquivos com mode("overwrite")

Vantagens sobre a alternativa (append com deduplicação):

A fonte é IMUTÁVEL e COMPLETA: a ANAC publica o mês completo e, quando tem necessidade de ajustes, republica o arquivo todo.
append exigiria uma chave de negócio para deduplicar. O VRA não tem uma chave nativa.
Overwrite é atômico, garantindo que ou a nova versão seja exibida integralmente ou a versão antiga ainda seja a válida.
Mantém o histórico. Cada overwrite gera uma versão nova e a anterior continua disponível por time travel

In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
    )

print(f"{TABELA}: {spark.table(TABELA).count():,} linhas ")

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS
          'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 à jul/2026).
           Dado Bruto: todas as colunas string, nenhuma linha descartada.
           Carga full refresh idempotente à partir de /Volumes/voebem/bronze/arquivos/vra/.'
          """)

In [0]:
display(
    spark.sql(f"""
              SELECT _arquivo_origem, COUNT(*) AS linhas, MAX
              (_ingerido_em) AS _ingerido_em
              FROM {TABELA}
              GROUP BY _arquivo_origem
              SORT BY _arquivo_origem
              """)
)